In [1]:
import pandas as pd

df = pd.read_csv("../datasets/clean_reviews.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (34615, 19)


C:\Users\d_fon\AppData\Local\Temp\ipykernel_13832\411630371.py:3: DtypeWarning: Columns (1: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../datasets/clean_reviews.csv")


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,review_length,sentiment
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,Adapter,27,positive
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,truman,14,positive
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,DaveZ,26,positive
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,Shacks,117,positive
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,explore42,117,positive


In [2]:
texts = df["reviews.text"]

print("Number of reviews:", len(texts))

Number of reviews: 34615


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_cluster = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))

X_tfidf_cluster = tfidf_cluster.fit_transform(texts)

print("TF-IDF shape:", X_tfidf_cluster.shape)

TF-IDF shape: (34615, 5000)


In [4]:
from sklearn.cluster import KMeans

k_values = [4, 5, 6]
kmeans_models = {}

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    
    kmeans.fit(X_tfidf_cluster)
    kmeans_models[k] = kmeans
    
    print(f"K={k} completed")

K=4 completed
K=5 completed
K=6 completed


In [5]:
from sklearn.metrics import silhouette_score

silhouette_results = {}

for k, model in kmeans_models.items():
    
    score = silhouette_score(X_tfidf_cluster, model.labels_, sample_size=5000, random_state=42)
    
    silhouette_results[k] = score
    
    print(f"K={k} | Silhouette Score: {score:.4f}")

K=4 | Silhouette Score: 0.0054
K=5 | Silhouette Score: 0.0059
K=6 | Silhouette Score: 0.0073


In [6]:
kmeans_6 = kmeans_models[6]

df["cluster"] = kmeans_6.labels_

df["cluster"].value_counts().sort_index()

cluster
0     2761
1    17011
2     1938
3     2765
4     4714
5     5426
Name: count, dtype: int64

In [7]:
import numpy as np

feature_names = tfidf_cluster.get_feature_names_out()

for cluster_num in range(6):
    center = kmeans_6.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-15:][::-1]
    top_terms = feature_names[top_indices]
    
    print(f"\nCluster {cluster_num}:")
    print(", ".join(top_terms))


Cluster 0:
easy, easy use, use, set, easy set, great, love, tablet, product, use great, tablet easy, good, love easy, setup, bought

Cluster 1:
great, love, product, good, echo, alexa, works, music, amazon, bought, use, like, kids, device, just

Cluster 2:
tv, amazon, amazon tv, stick, box, cable, apple, great, apple tv, 4k, movies, streaming, watch, love, shows

Cluster 3:
loves, old, year, year old, bought, daughter, tablet, son, gift, daughter loves, christmas, great, games, son loves, got

Cluster 4:
kindle, read, reading, books, love, light, paperwhite, great, screen, easy, reader, like, new, book, love kindle

Cluster 5:
tablet, great, price, good, great tablet, kids, amazon, tablet great, apps, tablet price, love, good tablet, games, great price, use


In [8]:
for cluster_num in range(6):
    print(f"\n--- Cluster {cluster_num} ---")
    
    top_products = (df[df["cluster"] == cluster_num]["name"].value_counts().head(5))
    
    print(top_products)


--- Cluster 0 ---
name
Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta                                       918
Echo (White),,,\r\nEcho (White),,,                                                                           254
All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta                      253
Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrome Paperwhite - touchscreen - Wi-Fi - black,,,    233
Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,                                                                       164
Name: count, dtype: int64

--- Cluster 1 ---
name
Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta                                       4037
Echo (White),,,\r\nEcho (White),,,                                                                           2742
Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,                                                                       2255
All-New Fire HD 8 T

In [9]:
for cluster_num in range(6):
    print(f"\n--- Cluster {cluster_num} sample reviews ---")
    
    samples = (df[df["cluster"] == cluster_num]["reviews.text"].sample(n=3, random_state=42))
    
    for review in samples:
        print("-", review[:300])


--- Cluster 0 sample reviews ---
- Device is preloaded with many friendly apps that are easy to use.
- Great product easy to use. I was very please due to the user interface
- Works great, easy to use for my 4 year old. I can adjust settings easily. Easy set up

--- Cluster 1 sample reviews ---
- I really love the tablets my kids love them.I recommend then to other people.
- 1 star...because it worked 1 time and stopped working.Returned it the very next day!My husband's Nexus charger and the Belkin 10W charger (that he got to replace this charger) do a GREAT job charging my Kindle Fire 8.9 HDX!
- I like it kids like it definitely needs a case so 4 stars 5 if it had case

--- Cluster 2 sample reviews ---
- It's perfect for me, my husband and my daughterthere are so many apps we downloaded, Sling, Nick Jr, Disney, CNN etc.If you're looking for the end all answer to cut your cable TV service, FireTv does not have things like live CNN, AMC and other cable stations (yet), but you could pur

In [10]:
kmeans_5 = kmeans_models[5]

feature_names = tfidf_cluster.get_feature_names_out()

for cluster_num in range(5):
    center = kmeans_5.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-15:][::-1]
    top_terms = feature_names[top_indices]

    print(f"\nCluster {cluster_num}:")
    print(", ".join(top_terms))


Cluster 0:
great, love, product, tv, amazon, bought, echo, works, alexa, good, music, use, like, gift, device

Cluster 1:
year, year old, old, loves, bought, bought year, tablet, old loves, great, old son, tablet year, games, old daughter, son, use

Cluster 2:
easy, easy use, use, set, easy set, great, love, tablet, product, use great, tablet easy, good, bought, setup, love easy

Cluster 3:
kindle, read, reading, books, love, light, paperwhite, great, screen, easy, reader, like, new, book, bought

Cluster 4:
tablet, price, great, good, great tablet, kids, great price, amazon, bought, tablet great, apps, love, games, good tablet, tablet price


In [11]:
import pandas as pd

pd.Series(kmeans_5.labels_).value_counts().sort_index()

0    18991
1     1186
2     2931
3     5029
4     6478
Name: count, dtype: int64

In [12]:
kmeans_4 = kmeans_models[4]

feature_names = tfidf_cluster.get_feature_names_out()

for cluster_num in range(4):
    center = kmeans_4.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-15:][::-1]
    top_terms = feature_names[top_indices]

    print(f"\nCluster {cluster_num}:")
    print(", ".join(top_terms))


Cluster 0:
great, love, kindle, product, amazon, tv, works, like, echo, good, use, alexa, music, device, just

Cluster 1:
tablet, price, great, good, great tablet, kids, great price, amazon, tablet great, apps, love, games, tablet price, good tablet, use

Cluster 2:
loves, bought, gift, old, year, year old, daughter, christmas, son, tablet, great, got, games, daughter loves, purchased

Cluster 3:
easy, easy use, use, set, easy set, great, love, tablet, product, use great, tablet easy, good, bought, setup, love easy


In [13]:
pd.Series(kmeans_4.labels_).value_counts().sort_index()

0    21282
1     6364
2     3925
3     3044
Name: count, dtype: int64

In [14]:
final_k = 6
final_kmeans = kmeans_models[final_k]

df["cluster"] = final_kmeans.labels_

print(df["cluster"].value_counts().sort_index())

cluster
0     2761
1    17011
2     1938
3     2765
4     4714
5     5426
Name: count, dtype: int64


- Clusters Profile

In [17]:
cluster_summary = (df.groupby("cluster").agg(reviews=("reviews.text", "count"), avg_rating=("reviews.rating", "mean")))

cluster_summary["percentage"] = (cluster_summary["reviews"] / len(df) * 100).round(2)

cluster_summary["avg_rating"] = (cluster_summary["avg_rating"].round(2))

cluster_summary

,reviews,avg_rating,percentage
cluster,,,
0,2761,4.75,7.98
1,17011,4.54,49.14
2,1938,4.71,5.60
3,2765,4.69,7.99
4,4714,4.67,13.62
5,5426,4.45,15.68


In [16]:
sentiment_by_cluster = pd.crosstab(df["cluster"], df["sentiment"], normalize="index").round(3) * 100

sentiment_by_cluster

sentiment,negative,neutral,positive
cluster,,,
0,0.3,0.8,98.9
1,3.1,4.9,92.0
2,1.4,2.9,95.7
3,0.8,2.6,96.6
4,1.9,3.0,95.1
5,2.5,6.8,90.6


In [18]:
cluster_names = {
    0: "Ease of Use & Setup",
    1: "General & Mixed Feedback",
    2: "Fire TV & Streaming",
    3: "Kids, Family & Gifts",
    4: "Kindle & Reading",
    5: "Tablets & Value"
}

df["cluster_name"] = df["cluster"].map(cluster_names)

df[[
    "reviews.text",
    "sentiment",
    "cluster",
    "cluster_name"
]].head()

,reviews.text,sentiment,cluster,cluster_name
0,This product so far has not disappointed. My c...,positive,1,General & Mixed Feedback
1,great for beginner or experienced person. Boug...,positive,3,"Kids, Family & Gifts"
2,Inexpensive tablet for him to use and learn on...,positive,5,Tablets & Value
3,I've had my Fire HD 8 two weeks now and I love...,positive,5,Tablets & Value
4,I bought this for my grand daughter when she c...,positive,5,Tablets & Value


## Final Conclusions — Clustering

The goal of this task was to discover meaningful groups of customer feedback using an unsupervised learning approach.

### Approach

- `reviews.text` was used as the main input for clustering.
- Text was transformed using TF-IDF with 5,000 features, English stop-word removal, and unigrams/bigrams.
- K-Means was tested with K=4, K=5, and K=6.
- The models were evaluated using both Silhouette Score and qualitative analysis of keywords, products, and sample reviews.

### Model Selection

Silhouette Scores:

- K=4: 0.0054
- K=5: 0.0059
- K=6: 0.0073

Although all silhouette scores were low, K=6 achieved the best score and produced the most interpretable semantic groups.

K=4 and K=5 generated larger general-purpose clusters that mixed several different product topics. K=6 provided better separation, particularly by identifying a clear Fire TV / Streaming group.

Therefore, K=6 was selected as the final clustering model.

### Identified Customer Feedback Themes

The six clusters were interpreted as:

- Cluster 0 — Ease of Use & Setup
- Cluster 1 — General & Mixed Feedback
- Cluster 2 — Fire TV & Streaming
- Cluster 3 — Kids, Family & Gifts
- Cluster 4 — Kindle & Reading
- Cluster 5 — Tablets & Value

The clusters represent customer feedback themes rather than strict product categories. Some groups are product-related, while others capture customer experience or usage context.

### Key Insights

The clusters show different customer experiences despite the dataset being strongly dominated by positive reviews.

- Ease of Use & Setup had the highest average rating (4.75) and 98.9% positive reviews.
- Fire TV & Streaming formed a clear topic around streaming, cable replacement, movies, and TV devices.
- Kindle & Reading showed strong semantic coherence around books, reading, Paperwhite, and e-readers.
- Kids, Family & Gifts captured reviews related to children, gifts, games, and family use.
- Tablets & Value had the lowest average rating (4.45) and the lowest positive share (90.6%) among the six clusters.
- General & Mixed Feedback was the largest cluster (49.14%) and contained more heterogeneous customer feedback.

### Limitations

The low Silhouette Scores indicate that the clusters are not strongly separated in the TF-IDF feature space.

This is consistent with the nature of customer reviews, where vocabulary and topics frequently overlap. The large General & Mixed Feedback cluster also shows that many reviews do not belong to one clearly defined topic.

For this reason, cluster interpretation was based not only on the numerical evaluation but also on top TF-IDF terms, dominant products, and real review examples.

Overall, the clustering provides useful thematic segmentation that can now be used as input for the GenAI summarization stage.

In [19]:
df.to_csv(
    "../datasets/clustered_reviews.csv",
    index=False
)

print("Clustered dataset saved!")
print(df.shape)

Clustered dataset saved!
(34615, 21)
